# Nordic Equity Risk Analyzer: analysis notebook

Screens Nordic large caps on **risk-adjusted return in SEK** and builds a long-only **max-Sharpe portfolio**.
The same maths powers the live web dashboard (`lib/metrics.ts`); `tests/fixture.json` keeps the two implementations
in sync.

**Questions this answers**
1. Which Nordic large caps delivered the best return *per unit of risk* over the period?
2. Does converting to SEK change the ranking? (Currency risk is real risk for a Swedish investor.)
3. How much did diversification help, and what does the in-sample optimal mix look like vs OMXS30?

In [ ]:
import pandas as pd
from nordic_screener import load_prices, load_universe, metrics_table, optimize, portfolio_path
from nordic_screener.metrics import format_table
from nordic_screener import plots

PERIOD = "5y"
RF = 0.02          # annual risk-free rate assumption
u = load_universe()
names = {s["ticker"]: s["name"] for s in u["stocks"]}
tickers = u["default_selection"]

## 1. Data
Adjusted closes (dividends reinvested) via `yfinance`, aligned across the four exchanges' holiday calendars, and converted to SEK with daily FX.

In [ ]:
prices, bench = load_prices(tickers, PERIOD, sek=True)
print(f"{prices.index[0].date()} to {prices.index[-1].date()} | {len(prices)} trading days")
prices.rename(columns=names).tail()

## 2. Screening: risk-adjusted ranking

In [ ]:
table = metrics_table(prices, bench, RF, names)
format_table(table)

## 3. Does currency matter?
Same screen in local currency. Differences in rank come purely from DKK/NOK/EUR moves against SEK.

In [ ]:
local, bench_l = load_prices(tickers, PERIOD, sek=False)
local_tbl = metrics_table(local, bench_l, RF, names)
cmp = pd.DataFrame({"Sharpe (SEK)": table["sharpe"], "Sharpe (local)": local_tbl["sharpe"]}).drop("OMXS30 (benchmark)")
cmp["Difference"] = cmp["Sharpe (SEK)"] - cmp["Sharpe (local)"]
cmp.sort_values("Difference").round(2)

## 4. Correlation

In [ ]:
returns = prices.pct_change().dropna()
plots.correlation_heatmap(returns.corr().rename(index=names, columns=names));

## 5. Portfolio optimisation
Monte Carlo sampling of long-only, fully invested portfolios (seeded and reproducible).

In [ ]:
opt = optimize(returns, RF)
single = opt["cloud"].iloc[: len(tickers)][["ret", "vol"]].set_axis([names[t] for t in tickers])
plots.efficient_frontier(opt, single)
pd.DataFrame({"Max Sharpe": opt["max_sharpe"]["weights"], "Min volatility": opt["min_vol"]["weights"]}) \
  .rename(index=names).sort_values("Max Sharpe", ascending=False).style.format("{:.1%}")

In [ ]:
paths = {
    "Max-Sharpe portfolio": portfolio_path(prices, opt["max_sharpe"]["weights"]),
    "Equal weight": portfolio_path(prices, opt["equal"]["weights"]),
    "OMXS30 (price index)": bench / bench.iloc[0],
}
plots.performance(paths);

## 6. Caveats
- **In-sample:** the max-Sharpe weights are chosen with hindsight on the same data they are scored on. A fair test would fit on one window and evaluate on the next (walk-forward).
- **Survivorship bias:** the universe is today's large caps.
- **Benchmark:** OMXS30 here is a price index (no dividends), while stocks are total return.
- No transaction costs, taxes or rebalancing frictions. Not investment advice.